<a href="https://colab.research.google.com/github/dorothea-pudding/114-1_TAICA_homework/blob/main/2.%20%E7%A5%9E%E7%B6%93%E7%B6%B2%E8%B7%AF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

我們終於要開始做生命中第一個神經網路。要做的是 3 層深度學習, 因此請自行設第一層 N1 神經元, 第二層 N2, 第三層 N3

In [ ]:
#各層神經元數量
N1 = 30
N2 = 30
N3 = 30
N4 = 30
N5 = 30

## 1. 讀入套件

這裡我們讀入一些套件, 今天暫時不要理會細節。

In [ ]:
!pip install gradio

In [ ]:
%matplotlib inline

# 標準數據分析、畫圖套件
import numpy as np #數值運算的基礎套件
import matplotlib.pyplot as plt #畫圖工具，畫圖表、顯示影像
from PIL import Image #影像處理，常用於載入、轉換、縮放影像

# 神經網路方面
import tensorflow as tf #google開發的深度學習框架，主要用於建議神經網路
from tensorflow.keras.datasets import mnist #內建的數字資料集（手寫數字案0～9，總共60000筆訓練資料）
from tensorflow.keras.utils import to_categorical #把數字標籤轉成＂oneone-hot eencoding＂，方便神經網路處理
from tensorflow.keras.models import Sequential #最常見的keras神經網路模型，層按照順序堆疊
from tensorflow.keras.layers import Dense #全連結層，每個神經元都跟前一層的神經元相連
from tensorflow.keras.optimizers import SGD #隨梯度下降，一種訓練神經網路的優化器

# 互動設計用
from ipywidgets import interact_manual
#ipywidgets：在jupyter notebook 建立互動式UI
#interact_manual：快速把一個python程式包裝成帶按鈕的UI

# 神速打造 web app 的 Gradio。常用於快速把機器學習模型做成demo
import gradio as gr

#詢問ChatGPT各套件功能

### 2.1 由 Keras 讀入 MNIST

讀入MNIST函式庫:
MNIST 是有一堆 0-9 的手寫數字圖庫。有 6 萬筆訓練資料, 1 萬筆測試資料。它是 "Modified" 版的 NIST 數據庫, 原來的版本有更多資料。這個 Modified 的版本是由 LeCun, Cortes, 及 Burges 等人做的。可以參考這個數據庫的原始網頁。

MNIST 可以說是 Deep Learning 最有名的範例, 它被 Deep Learning 大師 Hinton 稱為「機器學習的果蠅」。

Keras 很貼心的幫我們準備好 MNIST 數據庫, 我們可以這樣讀進來 (第一次要花點時間)。

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
#載入MNIST手寫資料集
#(訓練影像, 訓練標籤), (測試影像, 測試標籤)

In [ ]:
print(f'訓練資料總筆數為 {len(x_train)} 筆資料')
print(f'測試資料總筆數為 {len(x_test)} 筆資料')

### 2.2 數據庫的內容

每筆輸入 (x) 就是一個手寫的 0-9 中一個數字的圖檔, 大小為 28x28。而輸出 (y) 當然就是「正確答案」。我們來看看編訓練資料的 x 輸入、輸出的部份分別長什麼樣子。

In [ ]:
def show_xy(n=0):
    ax = plt.gca() #取得目前的座標軸，方便控制圖形
    X = x_train[n] #取出第n個訓練影像
    plt.xticks([], []) #隱藏x軸刻度
    plt.yticks([], []) #隱藏y軸刻度
    plt.imshow(X, cmap = 'Greys')
    print(f'本資料 y 給定的答案為: {y_train[n]}') #輸出標籤（答案）

In [ ]:
  interact_manual(show_xy, n=(0,59999)); #產生UI控制介面，（已定義的函式，n的參數滑桿範圍）。輸出是滑桿加按鈕，按了之後才顯示指定圖片
  #interact是自動版本，移動滑桿就會有輸出，會影響效率（ChatGPT的補充說明）

In [ ]:
def show_data(n = 100):
    X = x_train[n]
    print(X)

In [ ]:
interact_manual(show_data, n=(0,59999));

### 2.3 輸入格式整理

我們現在要用標準神經網路學學手寫辨識。原來的每筆數據是個 28x28 的矩陣 (array), 但標準神經網路只吃「平平的」, 也就是每次要 28x28=784 長的向量。因此我們要用 `reshape` 調校一下。

In [ ]:
#整理資料，一定要執行
x_train = x_train.reshape(60000, 784)/255
x_test = x_test.reshape(10000, 784)/255

### 2.4 輸出格式整理

我們可能會想, 我們想學的函數是這樣的型式:

$$\hat{f} \colon \mathbb{R}^{784} \to \mathbb{R}$$

其實這樣不太好! 為什麼呢? 比如說我們的輸入 x 是一張 0 的圖, 因為我們訓練的神經網路總會有點誤差, 所以可能會得到:

$$\hat{f}(x) = 0.5$$

那這意思是有可能是 0, 也有可能是 1 嗎!!?? 可是 0 和 1 根本不像啊。換句話說分類的問題這樣做其實不合理!

於是我們會做 "1-hot enconding", 也就是

* 1 -> [0, 1, 0, 0, 0, 0, 0, 0, 0]
* 5 -> [0, 0, 0, 0, 0, 1, 0, 0, 0]

等等。因為分類問題基本上都要做這件事, Keras 其實已幫我們準備好套件!

In [ ]:
#做標籤的one-hot編碼
y_train = to_categorical(y_train, 10) #(原始標籤, 類別總數)
y_test = to_categorical(y_test, 10)

我們來看看剛剛某號數據的答案。

In [ ]:
n = 87
y_train[n]

和我們想的一樣! 至此我們可以打造我們的神經網路了。

## 3. 打造第一個神經網路

我們決定了我們的函數是

$$\hat{f} \colon \mathbb{R}^{784} \to \mathbb{R}^{10}$$

這個樣子。而我們又說第一次要用標準神網路試試, 所以我們只需要再決定要幾個隱藏層、每層要幾個神經元, 用哪個激發函數就可以了。

### 3.1 決定神經網路架構、讀入相關套件

假如我們要用 ReLU 當激發函數, 要設計神經網路, 只差要指定多少個隱藏層、每層多少個神經元就好了!

設計完了基本上就是告訴 TensorFlow, 我們的想法就可以了!

### 3.2 建構我們的神經網路

和以前做迴歸或機器學習一樣, 我們就打開個「函數學習機」。標準一層一層傳遞的神經網路叫 `Sequential`, 於是我們打開一個空的神經網路。

In [ ]:
model = Sequential() #建立神經網路的骨架。做出一個神經網路的容器。可以用.add來不斷加層

我們每次用 `add` 去加一層, 從第一個隱藏層開始。而第一個隱藏層因為 TensorFlow 當然猜不到輸入有 784 個 features, 所以我們要告訴它。

In [ ]:
model.add(Dense(N1, input_dim=784, activation='relu')) #在模型中加一層（連結層類型（神經元數量, 指定輸入資料的維度, 用ReLU作為激活函數））

第二層開始就不用再說明輸入神經元個數 (因為就是前一層神經元數)。

In [ ]:
model.add(Dense(N2, activation='relu'))

In [ ]:
model.add(Dense(N3, activation='relu'))

In [ ]:
model.add(Dense(N4, activation='relu'))

In [ ]:
model.add(Dense(N5, activation='relu'))

輸出有 10 個數字, 所以輸出層的神經元是 10 個! 而如果我們的網路輸出是

$$(y_1, y_2, \ldots, y_{10})$$

我們還希望

$$\sum_{i=1}^{10} y_i = 1$$

這可能嗎, 結果是很容易, 就用 `softmax` 當激發函數就可以!!

In [ ]:
model.add(Dense(10, activation='softmax')) #softmax通常放在最後一層，把輸出的向量轉換成機率分布

至此我們的第一個神經網路就建好了!

### 3.3 組裝

和之前比較不一樣的是我們還要做 `compile` 才正式把我們的神經網路建好。你可以發現我們還需要做幾件事:

* 決定使用的 loss function, 一般是 `mse`
* 決定 optimizer, 我們用標準的 SGD
* 設 learning rate

為了一邊訓練一邊看到結果, 我們加設

    metrics=['accuracy']
    
本行基本上和我們的神經網路功能沒有什麼關係。

In [ ]:
model.compile(loss='mse', optimizer=SGD(learning_rate=0.087), metrics=['accuracy']) #設定神經網路的訓練方式（損失函數, 優化器, 評估指標）

## 4. 檢視我們的神經網路

我們可以檢視我們神經網路的架構, 可以確認一下是不是和我們想像的一樣。

### 4.1 看 model 的 summary

In [ ]:
model.summary() #快速查看神經網路結構

很快算算參數數目和我們想像是否是一樣的!

## 5. 訓練你的第一個神經網路

恭喜! 我們完成了第一個神經網路。現在要訓練的時候, 你會發現不是像以前沒頭沒腦把訓練資料送進去就好。這裡我們還有兩件事要決定:

* 一次要訓練幾筆資料 (`batch_size`), 我們就 100 筆調一次參數好了
* 這 6 萬筆資料一共要訓練幾次 (`epochs`), 我們訓練個 10 次試試

於是最精彩的就來了。你要有等待的心理準備...

In [ ]:
model.fit(x_train, y_train, batch_size=100, epochs=15) #開始訓練神經網路（訓練特徵，訓練標籤，每次梯度更新的資料數量，訓練次數）

## 6. 試用我們的結果

我們來用比較炫的方式來看看可愛的神經網路學習成果。對指令有問題可以參考《少年Py的大冒險：成為Python數據分析達人的第一門課》。

In [ ]:
loss, acc = model.evaluate(x_test, y_test) #評估訓練後的模型在測試資料的表現

In [ ]:
print(f"測試資料正確率 {acc*100:.2f}%")

我們 "predict" 放的是我們神經網路的學習結果。做完之後用 argmax 找到數值最大的那一項。

In [ ]:
predict = np.argmax(model.predict(x_test), axis=-1)
#ChatGPT對這行程式給我的詳細解釋：
#model.predict(x_test)對測試集的每筆資料做前向傳播，得到模型輸出的向量
#np.argmax(..., axis=-1）對每個向量取最大值的索引，代表預測的類別
#axis=-1沿著最後一個維度取最大值

In [ ]:
predict #ChatGPT的解釋：在colab這類互動環境的寫法，會直接顯示結果，但是一般的python腳本不會有反應

不要忘了我們的 `x_test` 每筆資料已經換成 784 維的向量, 我們要整型回 28x28 的矩陣才能當成圖形顯示出來!

In [ ]:
def test(測試編號):
    plt.imshow(x_test[測試編號].reshape(28,28), cmap='Greys') #顯示圖片
    print('神經網路判斷為:', predict[測試編號])

In [ ]:
interact_manual(test, 測試編號=(0, 9999));

到底測試資料總的狀況如何呢? 我們可以給我們神經網路「總評量」。

In [ ]:
score = model.evaluate(x_test, y_test)

In [ ]:
print('loss:', score[0])
print('正確率', score[1])

### 7. 用 Gradio 來展示

In [ ]:
def resize_image(inp):
    # 圖在 inp["layers"][0]
    image = np.array(inp["layers"][0], dtype=np.float32)
    image = image.astype(np.uint8)

    # 轉成 PIL 格式
    image_pil = Image.fromarray(image)

    # Alpha 通道設為白色, 再把圖從 RGBA 轉成 RGB
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3]) # 把圖片粘貼到白色背景上，使用透明通道作為遮罩
    image_pil = background

    # 轉換為灰階圖像
    image_gray = image_pil.convert("L")

    # 將灰階圖像縮放到 28x28, 轉回 numpy array
    img_array = np.array(image_gray.resize((28, 28), resample=Image.LANCZOS))

    # 配合 MNIST 數據集
    img_array = 255 - img_array

    # 拉平並縮放
    img_array = img_array.reshape(1, 784) / 255.0

    return img_array

In [ ]:
def recognize_digit(inp):
    img_array = resize_image(inp) #預處理輸入影像
    prediction = model.predict(img_array).flatten() #模型預測，攤平成一維
    labels = list('0123456789') #建立標籤清單
    return {labels[i]: float(prediction[i]) for i in range(10)} #回傳機率

In [ ]:
iface = gr.Interface(
    fn=recognize_digit,
    inputs=gr.Sketchpad(), #輸入方式 = 畫布
    outputs=gr.Label(num_top_classes=3), #辨識後的輸出，取最高的三個
    title="MNIST 手寫辨識", #標題
    description="請在畫板上繪製數字" #描述
)

iface.launch(share=True, debug=True) ##launch啟動web app(可以分享, 顯示完整debug訊息)